In [1]:
import os
import requests
from pathlib import Path
import random
from typing import List, Tuple
import zipfile
from zipfile import ZipFile

import torch
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt

try:
  from torchinfo import summary
except:
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from torchmetrics import Accuracy
except:
  !pip install -q torchmetrics
  from torchmetrics import Accuracy

try:
  from tqdm.auto import tqdm
except:
  !pip install -q tqdm.auto
  from tqdm.auto import tqdm

device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print("All imported")
print(f"Device: {device}")

/Users/admin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


All imported
Device: mps


In [2]:
def set_seeds(seed: int=42):
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

In [3]:
data_path = Path("data")
image_path = data_path / "pizza_steak_sushi"
zip_path = data_path / "pizza_steak_sushi.zip"

if image_path.is_dir():
  print(f"[INFO] {image_path} directory exists, skipping download.")
else:
  print(f"[INFO] Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)
  with open(zip_path, "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print(f"[INFO] Downloading {image_path.name} from github")
    f.write(request.content)
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    print(f"[INFO] Unzipping {image_path.name} data...")
    zip_ref.extractall(image_path)
    os.remove(zip_path)

[INFO] data/pizza_steak_sushi directory exists, skipping download.


In [4]:
train_data = image_path / "train"
test_data = image_path / "test"

train_data, test_data

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

In [5]:
# normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                  std=[0.229, 0.224, 0.225])

# manual_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     normalize
# ])

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT 
manual_transforms = weights.transforms()

print(f"Manually created transforms: {manual_transforms}")

train_dataset = datasets.ImageFolder(root=train_data,
                                     transform=manual_transforms,
                                     target_transform=None)
test_dataset = datasets.ImageFolder(root=test_data,
                                    transform=manual_transforms,
                                    target_transform=None)

class_names = train_dataset.classes
    
print("\n", len(train_dataset), len(test_dataset))
print(f"\n{class_names}")

Manually created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

 225 75

['pizza', 'steak', 'sushi']


In [6]:
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=BATCH_SIZE,
                             shuffle=False)

train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x1181f52b0>,
 <torch.utils.data.dataloader.DataLoader at 0x107f05670>)

In [7]:
set_seeds()

model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [8]:
set_seeds()

for param in model.features.parameters():
    param.requeries_grad = False 

model.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280,
              out_features=len(class_names),
              bias=True)).to(device)

In [9]:
summary(model=model,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
        )

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

In [10]:
import shutil
shutil.rmtree("runs", ignore_errors=True)

try:
    from torch.utils.tensorboard import SummaryWriter 
except:
    print("[INFO] Couldn`t find tensorboard... installing it.")
    !pip install -q tensorboard 
    from torch.utils.tensorboard import SummaryWriter 

writer = SummaryWriter()
writer

In [11]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
    model.train()
    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        loggits = model(X)
        loss = loss_fn(loggits, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        y_pred = torch.argmax(torch.softmax(loggits, dim=1), dim=1)
        train_acc += (y_pred == y).sum().item()/len(loggits)
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc 

In [12]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):
    test_loss, test_acc = 0, 0
    model.eval()

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            loggits = model(X)
            loss = loss_fn(loggits, y)
            test_loss += loss.item()
            test_pred = torch.argmax(loggits, dim=1)
            test_acc += (test_pred == y).sum().item()/len(loggits)
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc 

In [13]:
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.dataloader,
          test_dataloader: torch.utils.data.dataloader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          writer: torch.utils.tensorboard.writer.SummaryWriter):

    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []}

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        print(
            f"Epoch: {epoch+1} | "
            f"Train loss: {train_loss:.3f} | "
            f"Test loss: {test_loss:.3f} | "
            f"Train accuracy: {train_acc:.3f} | "
            f"Test accuracy: {test_acc:.3f} | "
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if writer:
            writer.add_scalars(main_tag="Loss",
                            tag_scalar_dict={"train_loss": train_loss,
                                                "test_loss": test_loss},
                                global_step=epoch)
            
            writer.add_scalars(main_tag="Accuracy",
                            tag_scalar_dict={"train_acc": train_acc,
                                                "test_acc": test_acc},
                                global_step=epoch)
            writer.close()
        else:
            pass

    return results

In [14]:
import gc

gc.collect()
torch.mps.empty_cache()

set_seeds()
NUM_EPOCHS = 5

model = model.to(device)

writer.add_graph(model=model,
                 input_to_model=torch.randn(1, 3, 224, 224).to(device))

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=NUM_EPOCHS,
                device=device,
                writer=writer)

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 0.748 | Test loss: 0.204 | Train accuracy: 0.645 | Test accuracy: 0.958 | 
Epoch: 2 | Train loss: 0.321 | Test loss: 0.506 | Train accuracy: 0.840 | Test accuracy: 0.856 | 
Epoch: 3 | Train loss: 0.174 | Test loss: 0.295 | Train accuracy: 0.977 | Test accuracy: 0.887 | 
Epoch: 4 | Train loss: 0.082 | Test loss: 0.371 | Train accuracy: 0.980 | Test accuracy: 0.836 | 
Epoch: 5 | Train loss: 0.408 | Test loss: 0.227 | Train accuracy: 0.871 | Test accuracy: 0.919 | 


In [15]:
results

{'train_loss': [0.747725885361433,
  0.3209376842714846,
  0.17408119305036962,
  0.08232223498634994,
  0.40815486339852214],
 'train_acc': [0.64453125, 0.83984375, 0.9765625, 0.98046875, 0.87109375],
 'test_loss': [0.20364352564016977,
  0.5063129961490631,
  0.2949673334757487,
  0.37053026755650836,
  0.22697336475054422],
 'test_acc': [0.9583333333333334,
  0.8560606060606061,
  0.8873106060606061,
  0.8361742424242425,
  0.9185606060606061]}

In [16]:
%load_ext tensorboard 
%tensorboard --logdir runs

In [17]:
def create_writer(experiment_name: str,
                  model_name: str,
                  extra: str=None):
    from datetime import datetime 
    timestamp = datetime.now().strftime("%Y-%m-%d")

    if extra:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)
    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")
    return SummaryWriter(log_dir=log_dir)

In [18]:
# example_writer = create_writer(experiment_name="data_10_percent",
#                                model_name="effnetb0",
#                                extra="5_epochs")

In [19]:
data_path = Path("data")
image_10_percent_data = data_path / "pizza_steak_sushi_10_percent"

if image_10_percent_data.is_dir() and len(os.listdir(image_10_percent_data)) > 0:
    print(f"{image_10_percent_data} directory exests, skipping downloading.")
else:
    print(f"Did not find {image_10_percent_data} directory, creating one...")
    image_10_percent_data.mkdir(parents=True, exist_ok=True)
    with open(data_path / "pizza_steak_sushi_10_percent.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print(f"Downloading pizza_steak_sushi_10_percent data in {image_10_percent_data} directory")
        f.write(request.content)
    with zipfile.ZipFile(data_path / "pizza_steak_sushi_10_percent.zip", "r") as zip_ref:
        print(f"Unzipping pizza_steak_sushi_10_percent data in {image_10_percent_data} directory")
        zip_ref.extractall(image_10_percent_data)

data_path = Path("data")
image_20_percent_data = data_path / "pizza_steak_sushi_20_percent"

if image_20_percent_data.is_dir() and len(os.listdir(image_20_percent_data)) > 0:
    print(f"{image_20_percent_data} directory exests, skipping downloading.")
else:
    print(f"Did not find {image_20_percent_data} directory, creating one...")
    image_20_percent_data.mkdir(parents=True, exist_ok=True)
    with open(data_path / "pizza_steak_sushi_20_percent.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip")
        print(f"Downloading pizza_steak_sushi_20_percent data in {image_20_percent_data} directory")
        f.write(request.content)
    with zipfile.ZipFile(data_path / "pizza_steak_sushi_20_percent.zip", "r") as zip_ref:
        print(f"Unzipping pizza_steak_sushi_20_percent data in {image_20_percent_data} directory")
        zip_ref.extractall(image_20_percent_data)

data/pizza_steak_sushi_10_percent directory exests, skipping downloading.
data/pizza_steak_sushi_20_percent directory exests, skipping downloading.


In [20]:
image_10_percent_data, image_20_percent_data

(PosixPath('data/pizza_steak_sushi_10_percent'),
 PosixPath('data/pizza_steak_sushi_20_percent'))

In [21]:
train_dir_10_percent = image_10_percent_data / "train"
train_dir_20_percent = image_20_percent_data / "train"
test_dir_10_percent = image_10_percent_data / "test"

print(f"Training directory 10%: {train_dir_10_percent}")
print(f"Training directory 20%: {train_dir_20_percent}")
print(f"Test directory 10%: {test_dir_10_percent}")

Training directory 10%: data/pizza_steak_sushi_10_percent/train
Training directory 20%: data/pizza_steak_sushi_20_percent/train
Test directory 10%: data/pizza_steak_sushi_10_percent/test


In [22]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

simple_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

In [23]:
train_data_10_percent = datasets.ImageFolder(root=train_dir_10_percent,
                                             transform=simple_transform,
                                             target_transform=None)
train_data_20_percent = datasets.ImageFolder(root=train_dir_20_percent,
                                             transform=simple_transform,
                                             target_transform=None)
test_data_10_percent = datasets.ImageFolder(
    root=test_dir_10_percent,
    transform=simple_transform,
    target_transform=None)

train_data_10_percent, train_data_20_percent, test_data_10_percent

(Dataset ImageFolder
     Number of datapoints: 225
     Root location: data/pizza_steak_sushi_10_percent/train
     StandardTransform
 Transform: Compose(
                Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
                ToTensor()
                Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ),
 Dataset ImageFolder
     Number of datapoints: 450
     Root location: data/pizza_steak_sushi_20_percent/train
     StandardTransform
 Transform: Compose(
                Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
                ToTensor()
                Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ),
 Dataset ImageFolder
     Number of datapoints: 75
     Root location: data/pizza_steak_sushi_10_percent/test
     StandardTransform
 Transform: Compose(
                Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
   

In [24]:
BATCH_SIZE = 32 

train_dataloader_10_percent = DataLoader(dataset=train_data_10_percent,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)
train_dataloader_20_percent = DataLoader(dataset=train_data_20_percent,
                                         batch_size=BATCH_SIZE,
                                         shuffle=True)
test_dataloader = DataLoader(dataset=test_data_10_percent,
                             batch_size=BATCH_SIZE,
                             shuffle=False)
class_names = train_data_10_percent.classes

train_dataloader_10_percent, train_dataloader_20_percent, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x11cded850>,
 ['pizza', 'steak', 'sushi'])

In [25]:
weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT 
model = torchvision.models.efficientnet_b2(weights=weights)

summary(model=model,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 1000]           --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1408, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

In [26]:
OUT_FEATURES = len(class_names)

def create_effnetb0():
    weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT 
    model = torchvision.models.efficientnet_b0(weights=weights).to(device)
    for param in model.features.parameters():
        param.requires_grad = False 
    set_seeds()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features=1280, out_features=OUT_FEATURES)
    ).to(device)
    model.name = "effnetb0"
    print(f"Created new {model.name} model.")
    return model

def create_effnetb2():
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT 
    model = torchvision.models.efficientnet_b2(weights=weights).to(device)
    for param in model.features.parameters():
        param.requires_grad = False 
    set_seeds()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features=1408, out_features=OUT_FEATURES)
    ).to(device)
    model.name = "effnetb2"
    print(f"Created new {model.name} model.")
    return model

In [27]:
effnetb0 = create_effnetb0()

summary(model=effnetb0,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Created new effnetb0 model.


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [28]:
effnetb2 = create_effnetb2()

summary(model=effnetb2,
        input_size=(32, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Created new effnetb2 model.


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   Partial
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1408, 7, 7]     --                   False
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   False
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   (864)                False
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   (64)                 False
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 

In [29]:
num_epochs = [5, 10]
models = ["effnetb0", "effnetb2"]
train_dataloaders = {"data_10_percent": train_dataloader_10_percent,
                     "data_20_percent": train_dataloader_20_percent}

In [30]:
def save_model(model_name: str,
              model: torch.nn.Module):
    PATH = Path("models")
    PATH.mkdir(parents=True, exist_ok=True)

    MODEL_NAME = model_name
    MODEL_SAVE_PATH = PATH / MODEL_NAME

    print(f"Saves weights model in {MODEL_SAVE_PATH}")
    torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)

In [31]:
set_seeds(seed=42)
experiment_number = 0 

for dataloader_name, train_dataloader in train_dataloaders.items():
    for epochs in num_epochs:
        for model_name in models:
            experiment_number += 1
            print(f"Experiment number: {experiment_number}")
            print(f"Model: {model_name}")
            print(f"Dataloader: {dataloader_name}")
            print(f"Number of epochs: {epochs}")

            if model_name == "effnetb0":
                model = create_effnetb0()
            else:
                model = create_effnetb2()

            loss_fn = nn.CrossEntropyLoss()
            optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

            train(model=model,
                  train_dataloader=train_dataloader,
                  test_dataloader=test_dataloader,
                  loss_fn=loss_fn,
                  optimizer=optimizer,
                  epochs=epochs,
                  device=device,
                  writer=create_writer(experiment_name=dataloader_name,
                                       model_name=model_name,
                                       extra=f"{epochs}_epochs"))
            save_filepath = f"{model_name}_{dataloader_name}_{epochs}_epochs.pth"
            save_model(model=model,
                       model_name=save_filepath)
            print("-"*50 + "\n")


Experiment number: 1
Model: effnetb0
Dataloader: data_10_percent
Number of epochs: 5
Created new effnetb0 model.
[INFO] Created SummaryWriter, saving to: runs/2026-10-07/data_10_percent/effnetb0/5_epochs...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 1.062 | Test loss: 0.934 | Train accuracy: 0.465 | Test accuracy: 0.457 | 
Epoch: 2 | Train loss: 0.866 | Test loss: 0.822 | Train accuracy: 0.645 | Test accuracy: 0.600 | 
Epoch: 3 | Train loss: 0.814 | Test loss: 0.712 | Train accuracy: 0.668 | Test accuracy: 0.866 | 
Epoch: 4 | Train loss: 0.704 | Test loss: 0.585 | Train accuracy: 0.754 | Test accuracy: 0.886 | 
Epoch: 5 | Train loss: 0.625 | Test loss: 0.555 | Train accuracy: 0.902 | Test accuracy: 0.897 | 
Saves weights model in models/effnetb0_data_10_percent_5_epochs.pth
--------------------------------------------------

Experiment number: 2
Model: effnetb2
Dataloader: data_10_percent
Number of epochs: 5
Created new effnetb2 model.
[INFO] Created SummaryWriter, saving to: runs/2026-10-07/data_10_percent/effnetb2/5_epochs...


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 1.082 | Test loss: 0.940 | Train accuracy: 0.359 | Test accuracy: 0.732 | 
Epoch: 2 | Train loss: 0.929 | Test loss: 0.859 | Train accuracy: 0.621 | Test accuracy: 0.814 | 
Epoch: 3 | Train loss: 0.821 | Test loss: 0.746 | Train accuracy: 0.723 | Test accuracy: 0.928 | 
Epoch: 4 | Train loss: 0.704 | Test loss: 0.714 | Train accuracy: 0.871 | Test accuracy: 0.928 | 
Epoch: 5 | Train loss: 0.654 | Test loss: 0.700 | Train accuracy: 0.785 | Test accuracy: 0.877 | 
Saves weights model in models/effnetb2_data_10_percent_5_epochs.pth
--------------------------------------------------

Experiment number: 3
Model: effnetb0
Dataloader: data_10_percent
Number of epochs: 10
Created new effnetb0 model.
[INFO] Created SummaryWriter, saving to: runs/2026-10-07/data_10_percent/effnetb0/10_epochs...


...

Epoch: 1 | Train loss: 0.984 | Test loss: 0.784 | Train accuracy: 0.529 | Test accuracy: 0.815 | 
Epoch: 2 | Train loss: 0.753 | Test loss: 0.668 | Train accuracy: 0.773 | Test accuracy: 0.907 | 
Epoch: 3 | Train loss: 0.608 | Test loss: 0.565 | Train accuracy: 0.842 | Test accuracy: 0.928 | 
Epoch: 4 | Train loss: 0.537 | Test loss: 0.573 | Train accuracy: 0.867 | Test accuracy: 0.928 | 
Epoch: 5 | Train loss: 0.450 | Test loss: 0.458 | Train accuracy: 0.900 | Test accuracy: 0.918 | 
Saves weights model in models/effnetb2_data_20_percent_5_epochs.pth
--------------------------------------------------

Experiment number: 7
Model: effnetb0
Dataloader: data_20_percent
Number of epochs: 10
Created new effnetb0 model.
[INFO] Created SummaryWriter, saving to: runs/2026-10-07/data_20_percent/effnetb0/10_epochs...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 0.951 | Test loss: 0.660 | Train accuracy: 0.617 | Test accuracy: 0.866 | 
Epoch: 2 | Train loss: 0.700 | Test loss: 0.595 | Train accuracy: 0.810 | Test accuracy: 0.887 | 
Epoch: 3 | Train loss: 0.569 | Test loss: 0.460 | Train accuracy: 0.850 | Test accuracy: 0.918 | 
Epoch: 4 | Train loss: 0.491 | Test loss: 0.439 | Train accuracy: 0.854 | Test accuracy: 0.928 | 
Epoch: 5 | Train loss: 0.490 | Test loss: 0.394 | Train accuracy: 0.840 | Test accuracy: 0.907 | 
Epoch: 6 | Train loss: 0.385 | Test loss: 0.368 | Train accuracy: 0.912 | Test accuracy: 0.928 | 
Epoch: 7 | Train loss: 0.386 | Test loss: 0.329 | Train accuracy: 0.875 | Test accuracy: 0.918 | 
Epoch: 8 | Train loss: 0.377 | Test loss: 0.337 | Train accuracy: 0.885 | Test accuracy: 0.938 | 
Epoch: 9 | Train loss: 0.324 | Test loss: 0.316 | Train accuracy: 0.929 | Test accuracy: 0.918 | 
Epoch: 10 | Train loss: 0.361 | Test loss: 0.281 | Train accuracy: 0.887 | Test accuracy: 0.918 | 
Saves weights model

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 0.984 | Test loss: 0.784 | Train accuracy: 0.529 | Test accuracy: 0.815 | 
Epoch: 2 | Train loss: 0.753 | Test loss: 0.668 | Train accuracy: 0.773 | Test accuracy: 0.907 | 
Epoch: 3 | Train loss: 0.608 | Test loss: 0.565 | Train accuracy: 0.842 | Test accuracy: 0.928 | 
Epoch: 4 | Train loss: 0.537 | Test loss: 0.573 | Train accuracy: 0.867 | Test accuracy: 0.928 | 
Epoch: 5 | Train loss: 0.450 | Test loss: 0.458 | Train accuracy: 0.900 | Test accuracy: 0.918 | 
Epoch: 6 | Train loss: 0.395 | Test loss: 0.468 | Train accuracy: 0.910 | Test accuracy: 0.918 | 
Epoch: 7 | Train loss: 0.363 | Test loss: 0.418 | Train accuracy: 0.925 | Test accuracy: 0.938 | 
Epoch: 8 | Train loss: 0.366 | Test loss: 0.432 | Train accuracy: 0.908 | Test accuracy: 0.887 | 
Epoch: 9 | Train loss: 0.307 | Test loss: 0.419 | Train accuracy: 0.925 | Test accuracy: 0.898 | 
Epoch: 10 | Train loss: 0.355 | Test loss: 0.394 | Train accuracy: 0.902 | Test accuracy: 0.928 | 
Saves weights model

In [32]:
%load_ext tensorboard
%tensorboard --logdir runs

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 6702), started 0:01:20 ago. (Use '!kill 6702' to kill it.)

In [35]:
best_model_path = "models/effnetb0_data_20_percent_10_epochs.pth"
best_model = create_effnetb0()
best_model.load_state_dict(torch.load(best_model_path))

Created new effnetb0 model.


<All keys matched successfully>

In [36]:
effnetb0_model_size = Path(best_model_path).stat().st_size // (1024*1024)
print(f"EfficientNetB0 feature extractall model size: {effnetb0_model_size} MB")

EfficientNetB0 feature extractall model size: 15 MB
